# Publication-Ready Analysis: CKA / SVD / Similarity Geometry

Minimal reproduction of three diagnostics comparing TAME/TAME-Fusion Vanilla vs.
pretrained-encoder (PT-S1) variants:

- **Task-aligned geometry** -- Spearman correlation of latent distance with label
  difference ($\rho(|\Delta y|, L2)$) and with structural similarity
  ($\rho(\text{Tanimoto}, \text{CosSim})$).
- **Modality reweighting** -- CKA(Fused, Graph) vs. CKA(Fused, Text_proj): does
  pretraining shift how much the fused representation resembles each modality?
- **Representation geometry** -- singular-value spectrum of the graph embedding, plus
  weight-rank vs. data-rank of the text projection.

This notebook is model-free: it loads small precomputed summary CSVs/NPZ instead of
the underlying trained checkpoints. Those numbers were extracted once (30 models x a
forward pass over the BACE test set) by running the CKA/SVD/correlation cells in the
full analysis notebook with `RUN_TRAINING=False` against the existing
`bace_pubready_v1_runs.pt` checkpoint bundle.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

data_dir = Path("data")
corr_df = pd.read_csv(data_dir / "pubready_corr_df.csv")
dom_df = pd.read_csv(data_dir / "pubready_dom_df.csv")
geom_df = pd.read_csv(data_dir / "pubready_geom_df.csv")
svd_curves = np.load(data_dir / "pubready_svd_graph_curves.npz")

## Panel A: Task-aligned geometry shift (Vanilla vs. PT-S1)

In [ ]:
figA, axA = plt.subplots(figsize=(7.2, 5.1))
axA.set_facecolor("white")

corr_mean = corr_df.groupby("model")[["rho_tanimoto_cos", "rho_deltaY_l2"]].mean()
families = [
    ("TAME", "TAME_VANILLA", "TAME_PT_S1"),
    ("TAME-Fusion", "TAME_FUSION_VANILLA", "TAME_FUSION_PT_S1"),
]
x_labels, vals_van, vals_pt = [], [], []
for fam_name, van_key, pt_key in families:
    x_labels.append(f"{fam_name}\n$\\rho(|\\Delta y|,L2)$")
    vals_van.append(float(corr_mean.loc[van_key, "rho_deltaY_l2"]))
    vals_pt.append(float(corr_mean.loc[pt_key, "rho_deltaY_l2"]))

    x_labels.append(f"{fam_name}\n$\\rho(Tanimoto,CosSim)$")
    vals_van.append(float(corr_mean.loc[van_key, "rho_tanimoto_cos"]))
    vals_pt.append(float(corr_mean.loc[pt_key, "rho_tanimoto_cos"]))

x = np.arange(len(x_labels))
w = 0.36
axA.bar(x - w / 2, vals_van, width=w, color="#9ca3af", alpha=0.9, label="Vanilla")
axA.bar(x + w / 2, vals_pt, width=w, color="#2563eb", alpha=0.9, label="PT-S1")
axA.set_xticks(x, x_labels, fontsize=9)
axA.set_ylabel("Spearman $\\rho$")
axA.grid(True, axis="y", ls="--", lw=0.5)
axA.legend(frameon=False)
plt.tight_layout()
plt.show()

## Panel B: Modality reweighting (CKA)

In [ ]:
figB, axB = plt.subplots(figsize=(7.2, 5.1))
axB.set_facecolor("white")

dom_mean = dom_df.groupby("model_key")[["cka_fused_graph", "cka_fused_textproj"]].mean()
order_b = ["TAME_FUSION_VANILLA", "TAME_FUSION_PT_S1", "TAME_VANILLA", "TAME_PT_S1"]
order_b = [k for k in order_b if k in dom_mean.index]
label_map_b = {
    "TAME_FUSION_VANILLA": "TFusion\nVanilla", "TAME_FUSION_PT_S1": "TFusion\nPT-S1",
    "TAME_VANILLA": "TAME\nVanilla", "TAME_PT_S1": "TAME\nPT-S1",
}

xb = np.arange(len(order_b))
wb = 0.38
axB.bar(xb - wb / 2, dom_mean.loc[order_b, "cka_fused_graph"].values, width=wb,
        color="#1d4ed8", alpha=0.88, label="CKA(Fused, Graph)")
axB.bar(xb + wb / 2, dom_mean.loc[order_b, "cka_fused_textproj"].values, width=wb,
        color="#f97316", alpha=0.88, label="CKA(Fused, Text_proj)")
axB.set_xticks(xb, [label_map_b[k] for k in order_b])
axB.set_ylabel("CKA")
axB.grid(True, axis="y", ls="--", lw=0.5)
axB.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## Panel D: SVD spectrum + weight-rank vs. data-rank (TAME family, graph embedding)

In [ ]:
figD, axD = plt.subplots(figsize=(7.2, 5.6))
axD.set_facecolor("white")

mv, sv = svd_curves["mean_vanilla"], svd_curves["std_vanilla"]
mp, sp = svd_curves["mean_pt_s1"], svd_curves["std_pt_s1"]

xv = np.arange(1, len(mv) + 1)
axD.semilogy(xv, mv, color="#9ca3af", lw=2.0, label="TAME Vanilla (graph)")
axD.fill_between(xv, np.clip(mv - sv, 1e-8, None), mv + sv, color="#9ca3af", alpha=0.20)

xp = np.arange(1, len(mp) + 1)
axD.semilogy(xp, mp, color="#2563eb", lw=2.0, label="TAME PT-S1 (graph)")
axD.fill_between(xp, np.clip(mp - sp, 1e-8, None), mp + sp, color="#2563eb", alpha=0.18)

axD.set_xlabel("Singular value index")
axD.set_ylabel("Normalized singular value (log scale)")
axD.grid(True, ls="--", lw=0.5)
axD.legend(frameon=False, fontsize=8, loc="upper right")

inset = axD.inset_axes([0.53, 0.08, 0.42, 0.42])
k_v, k_p = "TAME_VANILLA", "TAME_PT_S1"
subset = geom_df[geom_df["model_key"].isin([k_v, k_p])]
cap = subset.groupby("model_key")["eff_rank_weight"].mean().reindex([k_v, k_p])
util = subset.groupby("model_key")["eff_rank_textproj"].mean().reindex([k_v, k_p])
xi = np.arange(2)
wi = 0.34
inset.bar(xi - wi / 2, cap.values, width=wi, color="#93c5fd", alpha=0.95, label="Weight-rank")
inset.bar(xi + wi / 2, util.values, width=wi, color="#1d4ed8", alpha=0.95, label="Data-rank")
inset.set_xticks(xi, ["Vanilla", "PT-S1"], fontsize=8)
inset.set_ylabel("Rank", fontsize=8)
inset.tick_params(axis="y", labelsize=8)
inset.grid(True, axis="y", ls="--", lw=0.4)
inset.legend(frameon=False, fontsize=7, loc="upper left")

plt.tight_layout()
plt.show()